# BBO capstone — method and results walkthrough

Johan Landman · Imperial College PCMLAI · black-box optimisation (BBO) challenge

This notebook presents the method and results end to end, using only the data in this repository:

1. **Data** — the supplied initial design plus all 12 rounds of portal results.
2. **Progress** — best value found, round by round, for each of the eight functions.
3. **Surrogate** — the Gaussian-process (GP) model, its cross-validated settings, and which inputs matter.
4. **Noise** — what the deliberate repeat queries showed.
5. **Predictions** — the written prediction for each late-round query against the actual result.
6. **Final queries** — regenerating the round-13 submission from the data.

Run it from the `notebooks/` folder (`jupyter notebook BBO_capstone_walkthrough.ipynb`). It reads `../data/processed/`;
to rebuild that from the raw files first, run `python src/build_dataset.py` from the repository root.

In [1]:
import json, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, ConstantKernel, WhiteKernel
warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', None)

ROOT = Path('..')                      # repository root, relative to notebooks/
PROC = ROOT / 'data' / 'processed'
FUNCS = range(1, 9)

def load(i):
    """Full evaluation history of function i: source, round (0 = initial design), x1..xd, y."""
    return pd.read_csv(PROC / f'function_{i}.csv')

def xcols(d):
    return [c for c in d.columns if c.startswith('x')]

## 1. Data

Each function takes an input in the unit cube [0, 1]^d and returns one number to be **maximised**. The initial design was supplied by the programme; one query per function was added each round through the portal.

In [2]:
rows = []
for i in FUNCS:
    d = load(i); ini = d[d.source == 'initial']; por = d[d.source != 'initial']
    rows.append({'function': f'F{i}', 'd': len(xcols(d)), 'initial points': len(ini),
                 'portal queries': len(por), 'best initial y': ini.y.max(), 'best y found': d.y.max(),
                 'round of best': int(d.loc[d.y.idxmax(), 'round'])})
summary = pd.DataFrame(rows).set_index('function')
summary.style.format({'best initial y': '{:.4g}', 'best y found': '{:.4g}'})

,d,initial points,portal queries,best initial y,best y found,round of best
function,,,,,,
F1,2,10,12,7.711e-16,2,7
F2,2,10,12,0.6112,0.7257,9
F3,3,15,12,-0.03484,-0.02504,11
F4,4,30,12,-4.026,0.744,11
F5,4,20,12,1089,8662,7
F6,5,20,12,-0.7143,-0.06113,7
F7,6,30,12,1.365,2.39,12
F8,8,40,12,9.598,9.93,12


## 2. Progress — best value found by round

Each panel shows the best value found so far. Round 0 is the best point in the supplied data.

In [3]:
fig, axes = plt.subplots(2, 4, figsize=(15, 6.5))
for ax, i in zip(axes.ravel(), FUNCS):
    d = load(i)
    best = [d[d['round'] <= r].y.max() for r in range(0, int(d['round'].max()) + 1)]
    ax.plot(range(len(best)), best, marker='o', ms=4, color='#1f5fa8')
    ax.set_title(f'F{i} (d = {len(xcols(d))})'); ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True)); ax.set_xlabel('round'); ax.grid(alpha=.3)
axes[0, 0].set_ylabel('best y so far'); axes[1, 0].set_ylabel('best y so far')
fig.suptitle('Best value found by round, per function (round 0 = supplied data)')
fig.tight_layout()
fig.savefig(ROOT / 'docs' / 'images' / 'best_so_far.png', dpi=110)
plt.show()

<Figure: saved to docs/images/best_so_far.png and shown below>

![Best value found by round](../docs/images/best_so_far.png)

## 3. Surrogate model — GP settings and input relevance

Each round the GP's kernel family (RBF or Matérn-5/2) and lengthscale floor were chosen per function by 5-fold
cross-validation (`src/tune.py`). The model below uses the settings selected before the final round.
Each input has its own lengthscale (automatic relevance determination); a short lengthscale means the output
changes quickly along that input. The share of 1/lengthscale² is used as a simple relevance score.

In [4]:
S = json.load(open(ROOT / 'rounds' / 'week12' / 'gp_settings.json'))

def gp_for(i):
    """Fit the GP surrogate exactly as the round scripts do (F5 modelled on log y)."""
    d = load(i); X = d[xcols(d)].to_numpy(); y = d.y.to_numpy()
    z = np.log(y) if i == 5 else y
    s, k = S[str(i)], X.shape[1]
    base = (RBF(np.full(k, .3), (s['ls_lo'], 2.)) if s['kernel'] == 'rbf'
            else Matern(np.full(k, .3), (s['ls_lo'], 2.), nu=2.5))
    kern = ConstantKernel(1., (1e-3, 1e3)) * base + WhiteKernel(1e-4, (1e-8, 1.))
    return GaussianProcessRegressor(kernel=kern, normalize_y=True, n_restarts_optimizer=25,
                                    random_state=0).fit(X, z)

rel = {}
for i in FUNCS:
    g = gp_for(i)
    ls = np.atleast_1d(g.kernel_.k1.k2.length_scale)
    w = 1 / ls**2; rel[f'F{i}'] = {f'x{j+1}': v for j, v in enumerate(w / w.sum())}
    rel[f'F{i}']['kernel'] = S[str(i)]['kernel']; rel[f'F{i}']['CV RMSE'] = S[str(i)]['cv_rmse']
rel = pd.DataFrame(rel).T
rel = rel[[f'x{j}' for j in range(1, 9)] + ['kernel', 'CV RMSE']]
rel.style.format('{:.0%}', na_rep='', subset=[c for c in rel if c.startswith('x')]).format('{:.3f}', subset=['CV RMSE'])

,x1,x2,x3,x4,x5,x6,x7,x8,kernel,CV RMSE
F1,50%,50%,,,,,,,rbf,0.582
F2,98%,2%,,,,,,,rbf,0.158
F3,0%,0%,100%,,,,,,rbf,0.071
F4,20%,21%,37%,23%,,,,,matern,2.808
F5,5%,57%,11%,26%,,,,,rbf,1.310
F6,37%,23%,11%,20%,8%,,,,matern,0.288
F7,5%,10%,1%,22%,39%,23%,,,matern,0.183
F8,20%,9%,34%,8%,4%,4%,17%,4%,rbf,0.281


**Reading the table.** Most functions are effectively lower-dimensional than they look: F2 depends almost entirely on x1, F3 on x3, and three of F8's eight inputs carry most of the relevance. That is why late rounds moved one coordinate at a time along the inputs that matter.

## 4. Noise — what the repeat queries showed

Submitting exactly the same input twice is the only direct way to measure noise. Identical inputs are grouped below.

In [5]:
reps = []
for i in FUNCS:
    d = load(i); key = d[xcols(d)].round(6).astype(str).agg('-'.join, axis=1)
    for k, g in d.groupby(key):
        if len(g) > 1:
            reps.append({'function': f'F{i}', 'input': k, 'readings': ', '.join(f'{v:.4g}' for v in g.y),
                         'mean': g.y.mean(), 'sd': g.y.std()})
pd.DataFrame(reps).style.format({'mean': '{:.4g}', 'sd': '{:.3g}'})

,function,input,readings,mean,sd
0,F1,0.628541-0.628541,"2, 2, 2, 2, 2, 2",2,2.43e-16
1,F2,0.682637-0.926564,"0.6503, 0.7257, 0.5975, 0.4597",0.6083,0.112
2,F3,0.492581-0.611593-0.851634,"-0.0258, -0.04638",-0.03609,0.0146
3,F5,0.999999-0.999999-0.999999-0.999999,"8662, 8662, 8662, 8662, 8662",8662,0
4,F6,0.450943-0.365418-0.636307-0.737863-0.002595,"-0.06113, -0.08777",-0.07445,0.0188


F1 and F5 return identical values (noise-free); F2, F3 and F6 do not. On F2 the record of 0.726 sits at a point whose four readings average 0.608 — a single good reading on a noisy function is a hypothesis, not a result.

## 5. Written predictions against results

From round 6, every query carried a numerical prediction written before submission (recorded in each
`rounds/weekNN/weekNN.py` docstring). The last two scored rounds:

In [6]:
pred = pd.DataFrame([
    # round, function, move, prediction, actual
    (11, 'F2', 'x1 ridge vertex',            0.614,   0.463161),
    (11, 'F3', 'GP mean-max, radius 0.05',  -0.0335, -0.025038),
    (11, 'F4', 'x4 vertex',                  0.7431,  0.744037),
    (11, 'F6', 'x2 vertex',                 -0.048,  -0.146622),
    (11, 'F7', 'six-coordinate GP step',     2.4471,  2.202056),
    (11, 'F8', 'dim 4 step',                 9.9285,  9.925247),
    (12, 'F2', 'replicate best-mean point',  0.66,    0.459710),
    (12, 'F3', 'GP mean-max, radius 0.03',  -0.0304, -0.026111),
    (12, 'F4', 'x2 -0.005',                  0.73,    0.739171),
    (12, 'F6', 'x2 vertex',                 -0.073,  -0.096991),
    (12, 'F7', 'x2 only',                    2.3255,  2.390110),
    (12, 'F8', 'dim 8 -0.10',                9.9322,  9.930109),
], columns=['round', 'function', 'move', 'prediction', 'actual'])
pred['error'] = pred.actual - pred.prediction
pred.style.format({'prediction': '{:.4g}', 'actual': '{:.4g}', 'error': '{:+.3g}'})

,round,function,move,prediction,actual,error
0,11,F2,x1 ridge vertex,0.614,0.4632,-0.151
1,11,F3,"GP mean-max, radius 0.05",-0.0335,-0.02504,+0.00846
2,11,F4,x4 vertex,0.7431,0.744,+0.000937
3,11,F6,x2 vertex,-0.048,-0.1466,-0.0986
4,11,F7,six-coordinate GP step,2.447,2.202,-0.245
5,11,F8,dim 4 step,9.928,9.925,-0.00325
6,12,F2,replicate best-mean point,0.66,0.4597,-0.2
7,12,F3,"GP mean-max, radius 0.03",-0.0304,-0.02611,+0.00429
8,12,F4,x2 -0.005,0.73,0.7392,+0.00917
9,12,F6,x2 vertex,-0.073,-0.09699,-0.024


Single-coordinate moves on noise-free functions (F4, F7, F8) landed close to prediction; the six-coordinate F7 step and the curve fits on noisy F2 and F6 did not. That pattern drove the final-round policy.

## 6. Final queries — regenerating the round-13 submission

The final round is pure exploitation: there is no later round to benefit from exploring.
- **Noise-free functions** (F1, F4, F5, F7, F8): the input with the highest observed output.
- **Noisy functions** (F2, F3, F6): identical inputs are averaged and ranked by mean − sd/√n, with sd pooled from
  the repeated points, so a single lucky reading cannot beat a replicated one.

In [7]:
NOISY = {2, 3, 6}

def final_query(i):
    d = load(i); xc = xcols(d)
    d['key'] = d[xc].round(6).astype(str).agg('-'.join, axis=1)
    g = d.groupby('key').agg(mean=('y', 'mean'), n=('y', 'size'), best=('y', 'max'))
    if i in NOISY:
        sd = d.groupby('key').y.std().dropna()
        sd = float(np.sqrt((sd**2).mean())) if len(sd) else 0.0
        g['score'] = g['mean'] - sd / np.sqrt(g['n'])
    else:
        g['score'] = g['best']
    x = d.loc[d.key == g.score.idxmax(), xc].iloc[0].clip(1e-6, 1 - 1e-6)
    return '-'.join(f'{v:.6f}' for v in x)

final = pd.DataFrame({'function': list(FUNCS), 'query_string': [final_query(i) for i in FUNCS]})
submitted = pd.read_csv(ROOT / 'rounds' / 'week13' / 'Week13_queries.csv')
assert (final.query_string == submitted.query_string).all(), 'mismatch with the submitted file'
print('Regenerated queries match rounds/week13/Week13_queries.csv')
final

Regenerated queries match rounds/week13/Week13_queries.csv


,function,query_string
0,1,0.628541-0.628541
1,2,0.682637-0.926564
2,3,0.444791-0.562256-0.859194
3,4,0.415962-0.417500-0.363817-0.416634
4,5,0.999999-0.999999-0.999999-0.999999
5,6,0.450943-0.365418-0.636307-0.737863-0.002595
6,7,0.002673-0.261928-0.255873-0.290302-0.327805-0.603094
7,8,0.198086-0.192416-0.216581-0.138786-0.800003-0.350000-0.198013-0.793085


## Summary

| | |
|---|---|
| **Solved** | F1 (Gaussian peak, maximum 2.0) and F5 (optimum at the corner (1, 1, 1, 1), 8,662.4) |
| **Largest relative gains** | F4 from −4.03 to 0.744; F6 from −0.714 to −0.061; F5 ×8 |
| **Still improving at the end** | F7 (1.365 → 2.390) |
| **Limited by noise** | F2 and F3, where repeat readings vary as much as any local signal |

The documentation in `docs/` gives the datasheet (per-function decisions) and the model card (method,
performance, limitations and trade-offs).